# LightGBM: Predicting Hard-Drive Failure
**IST 707 Final Project**

This notebook trains and tunes a LightGBM model to predict whether a hard drive will fail in the next 30 days, using its SMART health readings from the last 30 days. LightGBM builds boosted trees like XGBoost, but grows each tree leaf by leaf and bins the values first, which makes it fast on large tables. It uses the same data, inputs and time-based folds as the other model notebooks, so they can be compared directly. It:

1. loads the training and calibration splits (the 2025 test split stays closed until final evaluation)
2. builds the model inputs, following the EDA's decisions
3. trains a first LightGBM model on default settings and checks it on the calibration split
4. tunes its settings with a randomized search over time-based folds
5. scores the tuned model once on the calibration split and saves its predictions

## 1. Libraries
Imports everything the notebook uses:
- **pandas / numpy:** loading and handling the data
- **scikit-learn:** the pipeline pieces (`Pipeline`, `ColumnTransformer`, `FunctionTransformer`, `OneHotEncoder`), the tuning tool (`RandomizedSearchCV`) and the scoring metrics
- **lightgbm:** the model (`LGBMClassifier`)
- **time / joblib:** timing the search and saving the models to files

In [1]:
import numpy as np
import pandas as pd
import time
import joblib
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder
from lightgbm import LGBMClassifier
from sklearn.model_selection import RandomizedSearchCV, cross_validate
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    brier_score_loss
)

## 2. Loading the Data
Loads the two splits used for training and checking. **The test split is not loaded.**

- **`modeling_train.parquet`:** Jan 2023 – Oct 2024. 169,827 drive-windows, 8,087 failures. Healthy windows were down-sampled to 20 per failure, so failures are 1 in 21.
- **`modeling_calib.parquet`:** Nov – Dec 2024. 297,009 drive-windows, 328 failures, at the real failure rate of about 1 in 905.

For each split the cell creates:
- **`X_..._raw`:** every column except the answer
- **`y_...`:** the answer, `label` (1 = the drive fails in the next 30 days)
- **`meta_...`:** serial number, window number and model. These are **not** model inputs; they're used to build the time-based folds.

In [2]:
train = pd.read_parquet("../../data/modeling/modeling_train.parquet")
calib = pd.read_parquet("../../data/modeling/modeling_calib.parquet")

X_train_raw, y_train = train.drop(columns="label"), train["label"].astype(int)
X_calib_raw, y_calib = calib.drop(columns="label"), calib["label"].astype(int)

meta_train = train[["serial_number", "window", "model"]].reset_index(drop=True)
meta_calib = calib[["serial_number", "window", "model"]].reset_index(drop=True)

print("Training shape:", X_train_raw.shape)
print("Calibration shape:", X_calib_raw.shape)

Training shape: (169827, 93)
Calibration shape: (297009, 93)


## 3. Building the Model Inputs
Uses the same inputs as the decision tree, random forest, SVM and XGBoost notebooks, so every model sees the same columns. This only sets rules; nothing is calculated from the data here.

- **`KEEP_RAW`:** the SMART attributes kept, as raw values only. The EDA showed the vendor-normalized values lose detail and partly encode the brand. S10 is left out because it is always 0.
- **`vendor()`:** reads the manufacturer (Seagate, Toshiba, WDC, HGST) from the model name.
- **`build_features()`:**
  - keeps the first, last and change values for each kept attribute
  - keeps S9 (power-on hours) as drive age
  - turns the unknown capacity of -1 into a blank (23 training rows)
  - keeps days observed and the vendor
- **`NUMERIC` / `CATEGORICAL`:** the lists of number columns and the one text column (vendor).
- **`features` / `vendor_onehot`:** wrap these rules so they can sit inside the pipeline. The one-hot encoder turns vendor into yes/no columns, using a fixed vendor list so train and calibration always get the same columns.

In [3]:
KEEP_RAW = [1, 3, 4, 5, 7, 12, 192, 193, 194, 197, 198, 199]   # raw only; S9 -> age, S10 always 0
VENDORS = ["Seagate", "Toshiba", "WDC", "HGST", "Other"]


def vendor(model):
    m = str(model)
    if m.startswith("ST"): return "Seagate"
    if m.startswith("TOSHIBA"): return "Toshiba"
    if m.startswith(("WDC", "WUH")): return "WDC"
    if m.startswith(("HGST", "Hitachi")): return "HGST"
    return "Other"

## This Function Keeps raw SMART values only. Drops useless columns. Calculates drive age through power-on hours and cleans the storage column.
def build_features(df):
    X = pd.DataFrame(index=df.index)
    for i in KEEP_RAW:
        for s in ("first", "last", "delta"):
            X[f"s{i}_{s}"] = df[f"smart_{i}_raw_{s}"].astype("float64")
    X["s9_last"] = df["smart_9_raw_last"].astype("float64")
    X["capacity_tb"] = (df["capacity_bytes"] / 1e12).round(0).where(df["capacity_bytes"] > 0)   # -1 = unknown -> NaN
    X["days_observed"] = df["days_observed"].astype("float64")
    X["vendor"] = df["model"].map(vendor)
    return X

NUMERIC = [f"s{i}_{s}" for i in KEEP_RAW for s in ("first", "last", "delta")] + \
          ["s9_last", "capacity_tb", "days_observed"]

CATEGORICAL = ["vendor"]

features = FunctionTransformer(build_features)
vendor_onehot = OneHotEncoder(categories=[VENDORS], handle_unknown="ignore", sparse_output=False)

## 4. Preprocessing Pipeline
Chains the preprocessing steps into one object, `lgbm_preprocess`, the same as XGBoost's:

1. **features:** applies `build_features` to the raw table
2. **columns:**
   - **numbers:** passed through unchanged. LightGBM handles blanks itself, learning at each split which side missing values should go, so they don't need filling in. Like any tree model, it only compares values against cutoffs, so the columns don't need scaling.
   - **vendor:** one-hot encoded

In [4]:
lgbm_preprocess = Pipeline([
    ("features", features),
    ("columns", ColumnTransformer([
        ("numeric", "passthrough", NUMERIC),
        ("vendor", vendor_onehot, CATEGORICAL),
    ], verbose_feature_names_out=False)),
]).set_output(transform="pandas")

## 5. First LightGBM Model (Before Tuning)
Builds the full model, preprocessing followed by LightGBM on its **default settings**: 100 trees, up to 31 leaves each, learning rate 0.1.

Four settings are added that don't change the model:
- **`subsample_freq=1`:** LightGBM ignores `subsample` (the share of rows each tree sees) unless this is set. The default `subsample` is 1.0, all rows, so the untuned model is unchanged, but it lets the search in section 6 try smaller values.
- **`n_jobs=1`:** each model uses one core, so the search can run several models at once without them competing for cores
- **`verbose=-1`:** hides LightGBM's training log
- **`random_state=42`:** makes results repeatable

In [5]:
lgbm_model = Pipeline([
    ("preprocessing", lgbm_preprocess),
    ("classifier", LGBMClassifier(
        subsample_freq = 1,
        n_jobs = 1,
        verbose = -1,
        random_state = 42
    ))
])

### 5.1 Training the First Model
Fits the whole pipeline on the training split. The fitted model is saved to `lgbm_base.joblib`.

In [6]:
lgbm_model.fit(X_train_raw, y_train)
joblib.dump(lgbm_model, "../artifacts/lgbm_base.joblib")

['../artifacts/lgbm_base.joblib']

### 5.2 Checking the First Model on the Calibration Split
Predicts a failure probability for every calibration window, then turns those probabilities into yes/no predictions at thresholds from 0.0 to 1.0. At each threshold it records:

- **PR-AUC (AP):** how well the model *ranks* failing drives above healthy ones, over all thresholds. This is our main metric, and random guessing scores 0.0011.
- **ROC-AUC:** another ranking score (0.5 = coin flip, 1.0 = perfect). It looks good easily when failures are rare, so it's secondary.
- **Precision:** of the drives flagged, the share that actually failed
- **Recall:** of the drives that failed, the share that were flagged
- **F1:** one number balancing precision and recall
- **Brier score:** how close the probabilities are to what actually happened (lower is better)

PR-AUC, ROC-AUC and the Brier score don't depend on the threshold, which is why they're the same in every row. Only precision, recall and F1 change.

**Result:**
- PR-AUC on calib is **0.1045**, about 95 times the 0.0011 base rate. That is the highest untuned score so far: the untuned decision tree scored 0.0701, the untuned random forest 0.0995 and the untuned logistic regression 0.0739.
- ROC-AUC is 0.890, about the same as logistic regression (0.888), but LightGBM ranks the top of the list much better, which is what PR-AUC measures.
- At 0.5, recall is 0.58 and precision is 0.063, so about 1 in 16 flagged windows actually fails, against 1 in 65 for logistic regression.
- The Brier score of 0.0071 is low because no class weighting is used, so the probabilities stay close to the real failure rate.

In [7]:
results=[]
y_prob = lgbm_model.predict_proba(X_calib_raw)[:, 1]

# Default decision threshold
for x in [0.0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1.0]:
    y_pred = (y_prob >= x).astype(int)

    results.append({
        "Threshold": x,
        "Model": "LightGBM",
        "PR-AUC (AP)": average_precision_score(y_calib, y_prob),
        "ROC-AUC": roc_auc_score(y_calib, y_prob),
        "Precision": precision_score(
            y_calib, y_pred, zero_division=0
        ),
        "Recall": recall_score(
            y_calib, y_pred, zero_division=0
        ),
        "F1": f1_score(y_calib, y_pred, zero_division=0),
        "Brier Score": brier_score_loss(y_calib, y_prob)
    })

results_df = pd.DataFrame(results)
display(results_df.round(4))

,Threshold,Model,PR-AUC (AP),ROC-AUC,Precision,Recall,F1,Brier Score
0,0.0,LightGBM,0.1045,0.8895,0.0011,1.0000,0.0022,0.0071
1,0.1,LightGBM,0.1045,0.8895,0.0219,0.7165,0.0424,0.0071
2,0.2,LightGBM,0.1045,0.8895,0.0339,0.6799,0.0646,0.0071
3,0.3,LightGBM,0.1045,0.8895,0.0442,0.6555,0.0829,0.0071
4,0.4,LightGBM,0.1045,0.8895,0.0528,0.6250,0.0973,0.0071
5,0.5,LightGBM,0.1045,0.8895,0.0632,0.5762,0.1139,0.0071
6,0.6,LightGBM,0.1045,0.8895,0.0774,0.5061,0.1342,0.0071
7,0.7,LightGBM,0.1045,0.8895,0.1026,0.4360,0.1661,0.0071
8,0.8,LightGBM,0.1045,0.8895,0.1378,0.3323,0.1948,0.0071
9,0.9,LightGBM,0.1045,0.8895,0.2115,0.1677,0.1871,0.0071


## 6. Tuning LightGBM
### 6.1 Search Setup and Time-Based Folds
Sets up the tuning:

- **`lgbm_model`:** the search reuses the model from section 5. It works on a fresh copy, so the first model is left untouched, and each combination overwrites the settings being searched.
- **`param_distributions`:** the settings to try:
  - `num_leaves`: the most leaves a tree can have, which sets how complex each tree is
  - `learning_rate`: how big each tree's correction is
  - `n_estimators`: how many trees are built
  - `min_child_samples`: the fewest windows a leaf can hold, so no decision rests on a handful of drives
  - `subsample`: the share of rows each tree sees
  - `colsample_bytree`: the share of columns each tree sees
  - `reg_lambda`: a penalty that keeps leaf values small
  - `scale_pos_weight`: extra weight on failures. 1 means none; train is already 20:1, so the range stops at 20.

  Together that's 72,000 combinations.
- **`time_folds()`:** builds 4 cross-validation folds **by time**, the same function as every other model notebook. Each fold trains on earlier months and validates on later months, with a one-window gap in between, because each label looks one window ahead.

In [8]:
param_distributions = {
    "classifier__num_leaves": [15, 31, 63, 127],
    "classifier__learning_rate": [0.01, 0.03, 0.05, 0.1, 0.2],
    "classifier__n_estimators": [100, 200, 400, 800],
    "classifier__min_child_samples": [10, 20, 50, 100, 200],
    "classifier__subsample": [0.6, 0.8, 1.0],
    "classifier__colsample_bytree": [0.6, 0.8, 1.0],
    "classifier__reg_lambda": [0, 0.1, 1, 10],
    "classifier__scale_pos_weight": [1, 2, 5, 10, 20],
}


## Creating a cross validation set.
def time_folds(meta, n_splits=4, gap=1):
    """Expanding-window folds by calendar window. Skips `gap` windows between train and validation,
    because a window's label looks one window ahead."""
    w = meta["window"].to_numpy()
    windows = np.sort(np.unique(w))
    blocks = np.array_split(windows, n_splits + 1)
    folds = []
    for k in range(1, n_splits + 1):
        valid_w = blocks[k]
        train_w = windows[windows < valid_w.min() - gap]
        folds.append((np.flatnonzero(np.isin(w, train_w)), np.flatnonzero(np.isin(w, valid_w))))
    return folds

cv_folds = time_folds(meta_train)
for k, (tr_idx, va_idx) in enumerate(cv_folds):
    print(f"fold {k}: train {len(tr_idx):,} rows -> validate {len(va_idx):,} rows")

fold 0: train 28,508 rows -> validate 37,027 rows
fold 1: train 64,596 rows -> validate 31,106 rows
fold 2: train 95,671 rows -> validate 32,768 rows
fold 3: train 128,057 rows -> validate 33,501 rows


### 6.2 Untuned Model on the Time Folds
Scores the untuned LightGBM model from section 5 on the same 4 time-based folds the search uses, with the same four metrics: precision, recall, F1 and PR-AUC. This is the "before" number for tuning.

**Why this is needed.** Section 5.2 scored the untuned model on the calibration split, where failures are about 1 in 905. The search scores on training folds, where they are 1 in 21. PR-AUC depends heavily on the failure rate, so the 5.2 score can't be compared with the search scores in 6.3. Scoring the untuned model on the same folds the search uses leaves tuning as the only difference between them.

- **`cross_validate`:** trains the untuned model once per fold, scores it on that fold's validation months, and reports the mean and std of each metric over the 4 folds.
- **`scoring`:** the same four metrics the search uses, so the before and after numbers are measured the same way.

**Result:**
- PR-AUC: **0.592** (std 0.034)
- Precision: 0.734 (std 0.014)
- Recall: 0.497 (std 0.032)
- F1: 0.592 (std 0.027)
- This is the baseline for the search. On the same folds the untuned decision tree scored 0.527 and the untuned logistic regression 0.532, so LightGBM starts well ahead.

In [9]:
scoring={
            "precision": "precision",
            "recall": "recall",
            "f1": "f1",
            "pr_auc": "average_precision",
        }

cv_untuned_results = cross_validate(lgbm_model, X_train_raw, y_train, cv = cv_folds, scoring = scoring, n_jobs = -1)

for i in cv_untuned_results:
    print(f'{i} mean: {np.mean(cv_untuned_results[i])}')
    print(f'{i} standard deviation: {np.std(cv_untuned_results[i])}')
    print(' ')

fit_time mean: 1.2652108073234558
fit_time standard deviation: 0.557138756070563
 
score_time mean: 0.2870502471923828
score_time standard deviation: 0.02591755525083302
 
test_precision mean: 0.7336815715156606
test_precision standard deviation: 0.014149378932773132
 
test_recall mean: 0.4971455135711245
test_recall standard deviation: 0.03203483256045698
 
test_f1 mean: 0.5923836470543735
test_f1 standard deviation: 0.027450392553311977
 
test_pr_auc mean: 0.591835273572131
test_pr_auc standard deviation: 0.03395756881756316
 


### 6.3 Randomized Search, Results and Saving the Best Model
- **Randomized search:** tries **20 random combinations** from the 72,000. Each is trained once per fold, giving **80 fits**, and scored by PR-AUC (`average_precision`).
- **Refit:** the best combination is retrained on all of the training data (`refit="pr_auc"`).
- **Results table:**
  - `PR_AUC`: the average PR-AUC over the 4 folds
  - `PR_AUC_std`: how much the 4 fold scores vary
  - `PR_AUC_rank`: 1 = best

  The table is saved to `lgbm_search_results_ranked.csv`.
- **Save:** the best model is saved to `lgbm_best.joblib`, so it can be reloaded without retraining.

**Result:**
- Best: **num_leaves 15, learning_rate 0.03, n_estimators 400, min_child_samples 10, subsample 0.8, colsample_bytree 0.6, reg_lambda 1, scale_pos_weight 1**
- Best PR-AUC: **0.5985** (std 0.029). The untuned model scored 0.5918 (std 0.034), so tuning gained 0.007. That is smaller than the fold std, so it is a small gain at best.
- The best setting uses smaller trees (15 leaves), a lower learning rate and more trees than the default. That is a slower, more regularised model.
- `scale_pos_weight` above 1 raises recall but lowers precision. The 5th-ranked setting reaches recall 0.67 with precision 0.47, and its PR-AUC is about the same.
- The 20 settings range from 0.537 to 0.598, so the choice of settings matters more here than for logistic regression.
- The search took 3.8 minutes.

These scores come from training folds, where failures are 1 in 21, so they're much higher than PR-AUC on the calibration split. Use them only to compare settings with each other and with section 6.2.

In [10]:
search = RandomizedSearchCV(
    estimator=lgbm_model,
    param_distributions=param_distributions,
    n_iter=20,
    scoring={
        "precision": "precision",
        "recall": "recall",
        "f1": "f1",
        "pr_auc": "average_precision",
    },
    cv=cv_folds,
    n_jobs=-1,
    refit="pr_auc",
    random_state=42,
    verbose=1,
    return_train_score=False,
)


start = time.time()
search.fit(X_train_raw, y_train)
print(f"\nSearch took {(time.time() - start) / 60:.1f} minutes")

cv_results = pd.DataFrame(search.cv_results_)

# All four metrics, averaged across the CV folds (column names follow the scoring keys)
results = pd.DataFrame({
    "PR_AUC":         cv_results["mean_test_pr_auc"],
    "PR_AUC_std":     cv_results["std_test_pr_auc"],
    "Precision":      cv_results["mean_test_precision"],
    "Precision_std":  cv_results["std_test_precision"],
    "Recall":         cv_results["mean_test_recall"],
    "Recall_std":     cv_results["std_test_recall"],
    "F1":             cv_results["mean_test_f1"],
    "F1_std":         cv_results["std_test_f1"],
    "mean_fit_time":  cv_results["mean_fit_time"],
})

for col in cv_results.columns:
    if col.startswith("param_"):
        results[col] = cv_results[col]

# Rank by PR-AUC (the same ranking the search used to pick the best model)
results["PR_AUC_rank"] = cv_results["rank_test_pr_auc"]
results = results.sort_values("PR_AUC_rank").reset_index(drop=True)

print("Best PR-AUC (time-CV on train):", round(search.best_score_, 4))
print("Best parameters:")
for k, v in search.best_params_.items():
    print(f"  {k}: {v}")

print("\nTop 10 models ranked by PR-AUC:")
display(results.head(10))

results.to_csv("lgbm_search_results_ranked.csv", index=False)

# Save the best model (chosen by PR-AUC, refit on all of train)
best_lgbm = search.best_estimator_
joblib.dump(best_lgbm, "../artifacts/lgbm_best.joblib")

Fitting 4 folds for each of 20 candidates, totalling 80 fits



Search took 3.8 minutes
Best PR-AUC (time-CV on train): 0.5985
Best parameters:
  classifier__subsample: 0.8
  classifier__scale_pos_weight: 1
  classifier__reg_lambda: 1
  classifier__num_leaves: 15
  classifier__n_estimators: 400
  classifier__min_child_samples: 10
  classifier__learning_rate: 0.03
  classifier__colsample_bytree: 0.6

Top 10 models ranked by PR-AUC:


,PR_AUC,PR_AUC_std,Precision,Precision_std,Recall,Recall_std,F1,F1_std,mean_fit_time,param_classifier__subsample,param_classifier__scale_pos_weight,param_classifier__reg_lambda,param_classifier__num_leaves,param_classifier__n_estimators,param_classifier__min_child_samples,param_classifier__learning_rate,param_classifier__colsample_bytree,PR_AUC_rank
0,0.598452,0.028616,0.741513,0.017053,0.490758,0.028897,0.590403,0.026119,4.052283,0.8,1,1.0,15,400,10,0.03,0.6,1
1,0.597094,0.033673,0.683090,0.014408,0.551796,0.044221,0.609826,0.032484,10.209317,1.0,2,0.1,63,800,10,0.01,0.6,2
2,0.591615,0.027828,0.679302,0.021671,0.556073,0.034323,0.611178,0.027435,8.468933,0.8,2,10.0,15,800,10,0.01,0.6,3
3,0.589811,0.033005,0.682094,0.015230,0.555985,0.040566,0.611986,0.028930,2.634329,1.0,2,1.0,127,100,50,0.05,1.0,4
4,0.589053,0.027969,0.466455,0.028685,0.669642,0.036378,0.549681,0.030382,3.872018,0.8,10,0.1,15,400,20,0.03,0.6,5
5,0.586076,0.036146,0.681992,0.021214,0.549806,0.043008,0.608436,0.034892,3.256939,1.0,2,0.1,15,400,50,0.10,1.0,6
6,0.585651,0.034992,0.583991,0.015246,0.614411,0.050738,0.598083,0.030179,5.451068,1.0,5,0.1,15,800,50,0.05,1.0,7
7,0.585313,0.030416,0.740493,0.015868,0.478013,0.029550,0.580716,0.026214,2.186653,0.8,1,10.0,63,100,50,0.03,1.0,8
8,0.584912,0.030163,0.370912,0.034804,0.693804,0.039713,0.483040,0.037736,11.389486,0.8,20,0.0,63,800,200,0.01,0.8,9
9,0.582379,0.037193,0.685054,0.019521,0.535616,0.041772,0.600664,0.032618,6.628103,1.0,2,10.0,15,800,50,0.10,0.8,10


['../artifacts/lgbm_best.joblib']

## 7. Final Scoring on the Calibration Split
Scores the tuned LightGBM model from 6.3 **once** on the calibration split and saves its predictions for the model comparison. Nothing is tuned here.

- **Why calib:** the search chose its settings on training folds, where failures are 1 in 21. The calibration split (November to December 2024) has failures at the real rate of about 1 in 905, comes after every training month, and was never used to fit or tune this model. Every model is scored on exactly the same rows, so the scores can be compared directly.
- **Why only once:** calib holds only 328 failures. Scoring it repeatedly and adjusting the model each time would tune the model to those 328 drives. The 2025 test split stays closed until the final model is chosen.
- **The model:** loaded from `../artifacts/lgbm_best.joblib`, the same file the final test will load.
- **Saved file:** `lgbm_calib_preds.csv` in this folder, one row per calibration window, with the columns `serial_number`, `window`, `label` and `prob`.

**Result:**
- PR-AUC on calib: **0.1089** (297,009 windows, 328 failures)
- The untuned model in 5.2 scored 0.1045, so tuning raised it by 0.0044, about 4%.
- On the same rows the tuned decision tree scored 0.0736 and the tuned logistic regression 0.0750. LightGBM is ahead of both.

In [11]:
lgbm_final = joblib.load("../artifacts/lgbm_best.joblib")

calib_scores = lgbm_final.predict_proba(X_calib_raw)[:, 1]

calib_preds = pd.DataFrame({
    'serial_number': meta_calib['serial_number'].copy(),
    'window': meta_calib['window'].copy(),
    'label': y_calib.to_numpy(),
    'prob': calib_scores
})

calib_preds.to_csv('lgbm_calib_preds.csv', index = False)

print(f"Average precision score: {average_precision_score(calib_preds['label'], calib_preds['prob'])}")
print(f'Row count: {len(calib_preds)}')
print(f"Failure count: {calib_preds['label'].sum()}")

Average precision score: 0.10885227061867486
Row count: 297009
Failure count: 328
